# 建筑能耗异常注入

基于 `fwwb01_realistic.csv` 真实波动数据，按建筑类型注入 10 类异常，并新增 `anomaly_status` 列（anomaly / normal）。

- **Office** → primaryspaceusage == "Office"
- **Teaching** → primaryspaceusage == "Education"
- **Library** → primaryspaceusage == "Public services"

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

DATA_DIR = Path.cwd() / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"
SRC_FILE = DATA_DIR / "fwwb01_realistic.csv"
OUT_FILE = DATA_DIR / "fwwb01_realistic_with_anomalies.csv"

df = pd.read_csv(SRC_FILE, low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
df["date"] = df["timestamp"].dt.date
df["anomaly_status"] = "normal"

rng = np.random.default_rng(42)

def mark_anomaly(idx_list):
    df.loc[df.index.isin(idx_list), "anomaly_status"] = "anomaly"

print(f"总行数: {len(df):,}")
print(df["primaryspaceusage"].value_counts())

总行数: 26,343
primaryspaceusage
Education          8783
Office             8783
Public services    8777
Name: count, dtype: int64


## 一、办公楼 (Office) 异常

In [2]:
# 辅助：取某建筑从某索引起连续 n 行的索引
def get_consecutive_indices(building_mask, start_idx, n):
    idx_all = df.index[building_mask].tolist()
    if start_idx not in idx_all:
        return []
    pos = idx_all.index(start_idx)
    return idx_all[pos : pos + n]

# ----- 1. 办公楼-周末排程失效：4 个周六 08:00 起 24h -----
office_mask = df["primaryspaceusage"] == "Office"
sat_8 = df.loc[office_mask & (df["dayofweek"] == 5) & (df["hour"] == 8), "date"].unique()
chosen = rng.choice(sat_8, size=min(4, len(sat_8)), replace=False)
for d in chosen:
    start_row = df[(office_mask) & (df["date"] == d) & (df["hour"] == 8)].head(1)
    if start_row.empty:
        continue
    idx_list = get_consecutive_indices(office_mask, start_row.index[0], 24)
    for i in idx_list:
        df.at[i, "hac_energy"] = 85.0
        df.at[i, "sim_supply_temp"] = 7.0
        df.at[i, "sim_return_temp"] = 7.5
        df.at[i, "anomaly_status"] = "anomaly"
print("1. 周末排程失效 已注入")

1. 周末排程失效 已注入


In [3]:
# ----- 2. 办公楼-掩蔽效应：6-9 月周五 16:00-18:00 随机 3 小时，基础电*0.7，空调*1.25，重算总电 -----
office_mask = df["primaryspaceusage"] == "Office"
summer_fri = df.loc[office_mask & (df["timestamp"].dt.month.isin([6,7,8,9])) & (df["dayofweek"] == 4) & (df["hour"] >= 16) & (df["hour"] < 19)]
if len(summer_fri) >= 30:
    starts = summer_fri.groupby("date").head(1).index.tolist()
    picks = rng.choice(starts, size=min(12, len(starts)), replace=False)
    for start_idx in picks:
        idx_list = get_consecutive_indices(office_mask, start_idx, 3)
        for i in idx_list:
            base_elec = df.at[i, "electricity"] - df.at[i, "hac_energy"]
            df.at[i, "hac_energy"] = df.at[i, "hac_energy"] * 1.25
            df.at[i, "electricity"] = base_elec * 0.7 + df.at[i, "hac_energy"]
            df.at[i, "anomaly_status"] = "anomaly"
print("2. 掩蔽效应 已注入")

2. 掩蔽效应 已注入


In [4]:
# ----- 3. 办公楼-大流量小温差：制冷季(气温>26℃)随机 2 天，48h 回水=供水+1.5℃，hac*1.35 -----
office_mask = df["primaryspaceusage"] == "Office"
cooling = df.loc[office_mask & (df["airTemperature"] > 26)].groupby("date").head(1)
if len(cooling) >= 2:
    starts = rng.choice(cooling.index.tolist(), size=min(3, len(cooling)), replace=False)
    for start_idx in starts:
        idx_list = get_consecutive_indices(office_mask, start_idx, 48)
        for i in idx_list:
            if df.at[i, "hac_energy"] > 0.1:
                df.at[i, "sim_return_temp"] = df.at[i, "sim_supply_temp"] + 1.5
                df.at[i, "hac_energy"] = df.at[i, "hac_energy"] * 1.35
                df.at[i, "anomaly_status"] = "anomaly"
print("3. 大流量小温差 已注入")

3. 大流量小温差 已注入


## 二、教学楼 (Education) 异常

In [5]:
# ----- 4. 教学楼-管网跑水：随机 10 次，每次 N 小时 water*1.5~2.0 且 +>1.0 -----
edu_mask = df["primaryspaceusage"] == "Education"
candidates = df.loc[edu_mask].groupby("date").head(1).index.tolist()
if len(candidates) >= 10:
    starts = rng.choice(candidates, size=10, replace=False)
    for start_idx in starts:
        n_h = int(rng.integers(2, 6))
        idx_list = get_consecutive_indices(edu_mask, start_idx, n_h)
        mult = float(rng.uniform(1.5, 2.0))
        for i in idx_list:
            df.at[i, "water"] = df.at[i, "water"] * mult + float(rng.uniform(1.0, 2.0))
            df.at[i, "anomaly_status"] = "anomaly"
print("4. 管网跑水 已注入")

4. 管网跑水 已注入


In [6]:
# ----- 5. 教学楼-大开窗：气温 28~32℃ 连续 4h，hac*1.4，回水+3~5℃ -----
edu_mask = df["primaryspaceusage"] == "Education"
hot = df.loc[edu_mask & (df["airTemperature"] >= 28) & (df["airTemperature"] <= 32)].groupby("date").head(1)
if len(hot) >= 20:
    starts = rng.choice(hot.index.tolist(), size=min(22, len(hot)), replace=False)
    for start_idx in starts:
        idx_list = get_consecutive_indices(edu_mask, start_idx, 4)
        delta = rng.uniform(3, 5)
        for i in idx_list:
            df.at[i, "hac_energy"] *= 1.4
            df.at[i, "sim_return_temp"] = df.at[i, "sim_return_temp"] + delta
            df.at[i, "anomaly_status"] = "anomaly"
print("5. 大开窗 已注入")

5. 大开窗 已注入


In [7]:
# ----- 6. 教学楼-主机效能衰退：airTemp>36℃ 的 13:00-16:00，hac 放大，供水 9.5~11℃ -----
edu_mask = df["primaryspaceusage"] == "Education"
extreme = df.loc[edu_mask & (df["airTemperature"] > 36) & (df["hour"] >= 13) & (df["hour"] <= 16)]
for i in extreme.index:
    df.at[i, "hac_energy"] *= rng.uniform(1.3, 1.6)
    df.at[i, "sim_supply_temp"] = rng.uniform(9.5, 11.0)
    df.at[i, "anomaly_status"] = "anomaly"
print("6. 主机效能衰退 已注入")

6. 主机效能衰退 已注入


## 三、图书馆 (Public services) 异常

In [8]:
# ----- 7. 图书馆-极端除湿：dewTemp>22℃ 连续 20h，供水 5.5℃，hac*2 -----
lib_mask = df["primaryspaceusage"] == "Public services"
humid = df.loc[lib_mask & (df["dewTemperature"] > 22)].groupby("date").head(1)
if len(humid) >= 4:
    starts = rng.choice(humid.index.tolist(), size=min(5, len(humid)), replace=False)
    for start_idx in starts:
        idx_list = get_consecutive_indices(lib_mask, start_idx, 20)
        for i in idx_list:
            df.at[i, "sim_supply_temp"] = 5.5
            df.at[i, "hac_energy"] = df.at[i, "hac_energy"] * 2.0
            df.at[i, "anomaly_status"] = "anomaly"
print("7. 极端除湿 已注入")

7. 极端除湿 已注入


In [9]:
# ----- 8. 图书馆-闭馆未关灯：夜间 00:00 起 8 次，连续 7h 共增加 50~80 kWh -----
lib_mask = df["primaryspaceusage"] == "Public services"
night = df.loc[lib_mask & (df["hour"] == 0)].groupby("date").head(1)
if len(night) >= 8:
    starts = rng.choice(night.index.tolist(), size=min(9, len(night)), replace=False)
    for start_idx in starts:
        idx_list = get_consecutive_indices(lib_mask, start_idx, 7)
        add_total = float(rng.uniform(50, 80))
        add_per_h = add_total / len(idx_list) if idx_list else 0
        for i in idx_list:
            df.at[i, "electricity"] = df.at[i, "electricity"] + add_per_h
            df.at[i, "anomaly_status"] = "anomaly"
print("8. 闭馆未关灯 已注入")

8. 闭馆未关灯 已注入


In [10]:
# ----- 9. 图书馆-过渡季阀门内漏：1~2 次，120h，hac 40~60 kWh，供水缓慢降 -----
lib_mask = df["primaryspaceusage"] == "Public services"
mild = df.loc[lib_mask & (df["airTemperature"] >= 18) & (df["airTemperature"] <= 24) & (df["hac_energy"] < 5)].groupby("date").head(1)
if len(mild) >= 2:
    starts = rng.choice(mild.index.tolist(), size=min(2, len(mild)), replace=False)
    for start_idx in starts:
        idx_list = get_consecutive_indices(lib_mask, start_idx, 120)
        base_hac = rng.uniform(40, 60)
        for k, i in enumerate(idx_list):
            df.at[i, "hac_energy"] = base_hac + rng.uniform(-5, 5)
            df.at[i, "sim_supply_temp"] = max(5, 12 - k * 0.05 + rng.uniform(-0.5, 0.5))
            df.at[i, "anomaly_status"] = "anomaly"
print("9. 过渡季阀门内漏 已注入")

9. 过渡季阀门内漏 已注入


In [11]:
# ----- 10. 图书馆-传感器漂移：5~8 次随机 1~3h，仅 sim_supply_temp 改为 3.5 或 18 -----
lib_mask = df["primaryspaceusage"] == "Public services"
any_row = df.loc[lib_mask].index.tolist()
for _ in range(rng.integers(5, 9)):
    start_idx = rng.choice(any_row)
    n_h = rng.integers(1, 4)
    idx_list = get_consecutive_indices(lib_mask, start_idx, n_h)
    temp_val = rng.choice([3.5, 18.0])
    for i in idx_list:
        df.at[i, "sim_supply_temp"] = temp_val
        df.at[i, "anomaly_status"] = "anomaly"
print("10. 传感器漂移 已注入")

10. 传感器漂移 已注入


In [12]:
# 保存：去掉辅助列 date，保留 anomaly_status
df_out = df.drop(columns=["date"], errors="ignore")
df_out.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
n_anom = (df_out["anomaly_status"] == "anomaly").sum()
print(f"已保存: {OUT_FILE}")
print(f"异常行数: {n_anom:,} / {len(df_out):,} ({100*n_anom/len(df_out):.2f}%)")
df_out["anomaly_status"].value_counts()

已保存: c:\Users\86156\Downloads\fwwb_data\建筑资源data\building-data-genome-project-2-v1.0\buds-lab-building-data-genome-project-2-3d0cbaf\data\fwwb01_realistic_with_anomalies.csv
异常行数: 1,086 / 26,343 (4.12%)


anomaly_status
normal     25257
anomaly     1086
Name: count, dtype: int64

In [13]:
# 异常检测与效果评估模块
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.cwd() / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"

# 读入额定值与真实值（含已注入的异常标签）
rated = pd.read_csv(DATA_DIR / "fwwb01_simulated.csv", low_memory=False)
real = pd.read_csv(DATA_DIR / "fwwb01_realistic_with_anomalies.csv", low_memory=False)

# 对齐、合并（按 timestamp + building_id）
rated["timestamp"] = pd.to_datetime(rated["timestamp"])
real["timestamp"] = pd.to_datetime(real["timestamp"])

cols_keep = [
    "timestamp", "building_id", "primaryspaceusage", "sqm",
    "airTemperature", "dewTemperature", "hour", "dayofweek",
    "water", "electricity", "hac_energy",
    "sim_supply_temp", "sim_return_temp",
    "anomaly_status",
]
real = real[cols_keep].copy()

df = pd.merge(
    rated[["timestamp", "building_id", "water", "electricity", "hac_energy"]],
    real,
    on=["timestamp", "building_id"],
    suffixes=("_rated", "_real"),
    how="inner",
)

# 避免除零
eps = 1e-6
for col in ["water", "electricity", "hac_energy"]:
    df[f"{col}_rated_safe"] = df[f"{col}_rated"].clip(lower=eps)

# ========= 第一层：并行异常判定 =========
# 1) 水耗异常
delta_water = df["water_real"] - df["water_rated"]
rel_water = delta_water / df["water_rated_safe"]
water_anom = (rel_water > 0.20) & (delta_water > 1.0)

# 2) 总电耗异常
delta_ele = df["electricity_real"] - df["electricity_rated"]
rel_ele = delta_ele / df["electricity_rated_safe"]
# “基础负荷的 10%” 这里用额定电耗的 10% 近似
abs_ele_thr = 0.10 * df["electricity_rated_safe"]
ele_anom = (rel_ele > 0.15) & (delta_ele > abs_ele_thr)

# 3) 空调能耗异常
delta_hac = df["hac_energy_real"] - df["hac_energy_rated"]
rel_hac = delta_hac / df["hac_energy_rated_safe"]
hac_anom = (rel_hac > 0.20) & (delta_hac > 20.0)

df["water_anomaly_flag"] = water_anom
df["electricity_anomaly_flag"] = ele_anom
df["hac_anomaly_flag"] = hac_anom

# 总体“检测为异常”的标记（任一子系统触发即认为该小时异常）
df["detected_anomaly"] = df[["water_anomaly_flag", "electricity_anomaly_flag", "hac_anomaly_flag"]].any(axis=1)

# ========= 第二层：空调根因诊断（仅对 hac_anomaly_flag=True 的行） =========
def diagnose_hvac(row):
    if not row["hac_anomaly_flag"]:
        return "none"
    # 供回水偏差
    supply = row["sim_supply_temp"]
    ret = row["sim_return_temp"]
    dT = ret - supply

    # 营业时间判断：这里简单认为 8-18 为营业，其余为非营业
    is_off_time = (row["hour"] < 8) | (row["hour"] > 18)

    # 诊断 A：主机效能低（供冷达不到设定点）
    # 条件：hac 偏高 + 供水偏离期望 > 1.5℃（这里无法直接比较期望，只用绝对偏高近似）
    if (supply > 8.5) or (supply < 5.0):
        return "A_low_efficiency"

    # 诊断 B：大流量小温差（dT < 3℃ 且 hac 偏高）
    if dT < 3.0:
        return "B_small_deltaT"

    # 诊断 C：非预期开机（非营业时间 + hac 明显偏高）
    if is_off_time and (row["hac_energy_real"] > row["hac_energy_rated"] + 10.0):
        return "C_unexpected_on"

    return "other_hvac"

df["hvac_root_cause"] = df.apply(diagnose_hvac, axis=1)

# ========= 与构建标签对比评估 =========
true_anom = df["anomaly_status"] == "anomaly"
pred_anom = df["detected_anomaly"]

tp = int(((pred_anom) & (true_anom)).sum())
fp = int(((pred_anom) & (~true_anom)).sum())
fn = int((~pred_anom & true_anom).sum())
tn = int((~pred_anom & ~true_anom).sum())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("=== 异常检测整体效果 ===")
print(f"总样本数: {len(df):,}")
print(f"真实异常数 (label=anomaly): {true_anom.sum():,}")
print(f"检测为异常数: {pred_anom.sum():,}")
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

print("\n=== 子系统触发统计（只看真实异常行）===")
print(df.loc[true_anom, ["water_anomaly_flag", "electricity_anomaly_flag", "hac_anomaly_flag"]].mean())

print("\n=== HVAC 根因诊断分布（限 hac_anomaly_flag=True）===")
print(df.loc[df["hac_anomaly_flag"], "hvac_root_cause"].value_counts())

=== 异常检测整体效果 ===
总样本数: 26,349
真实异常数 (label=anomaly): 1,088
检测为异常数: 1,010
TP=953, FP=57, FN=135, TN=25204
Precision=0.944, Recall=0.876, F1=0.908

=== 子系统触发统计（只看真实异常行）===
water_anomaly_flag          0.030331
electricity_anomaly_flag    0.065257
hac_anomaly_flag            0.787684
dtype: float64

=== HVAC 根因诊断分布（限 hac_anomaly_flag=True）===
hvac_root_cause
A_low_efficiency    482
B_small_deltaT      184
other_hvac          148
C_unexpected_on      43
Name: count, dtype: int64
